# GeoLatent — Colab Quick Test

Tests all major features of the `geolatent` package.
Installs directly from GitHub — no PyPI needed.

> **Run Cell 1 first, then restart the runtime, then run all remaining cells.**

In [ ]:
# Cell 1 — Install (run once, then restart runtime)
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip"] + list(args), check=True)

print("[1/4] Upgrading NumPy / SciPy / Plotly ...")
pip("install", "-q", "--upgrade", "numpy>=1.23", "scipy>=1.9", "plotly>=5.13")

print("[2/4] Reinstalling scikit-learn (ABI match) ...")
pip("install", "-q", "--force-reinstall", "--no-deps", "scikit-learn>=1.5")

print("[3/4] Installing umap-learn ...")
pip("install", "-q", "umap-learn>=0.5.3")

print("[4/4] Installing GeoLatent from GitHub ...")
pip("install", "-q", "git+https://github.com/FirePheonix/geolatent.git")

print("\nDone. NOW RESTART THE RUNTIME (Runtime > Restart runtime), then run from Cell 2.")

In [ ]:
# Cell 2 — Imports & renderer setup
import numpy as np
import plotly.io as pio
from sklearn.datasets import load_wine, load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC

import geolatent as gl
from geolatent import visualize_decision_geometry, inspect_latent_space, DARK_SCIENTIFIC

pio.renderers.default = "colab"
print(f"GeoLatent v{gl.__version__} ready.")

---
## Test 1 — Wine SVM · PCA (data-driven axes)
Checks: PCA projection, decision surfaces, confidence shells, scatter, centroids, ellipsoids.

In [ ]:
wine = load_wine()
svm = SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=0)
svm.fit(wine.data, wine.target)

fig = visualize_decision_geometry(
    model=svm,
    X=wine.data,
    y=wine.target,
    projection_method="pca",
    feature_names=list(wine.feature_names),
    class_names=dict(enumerate(wine.target_names)),
    show_confidence=True,
    show_centroids=True,
    show_ellipsoids=True,
    title="Test 1 — Wine SVM (PCA)",
)
fig.show()

---
## Test 2 — Wine SVM · Sensitivity (model-driven axes)
Checks: finite-difference Jacobian projection, named sensitivity axes, inverse-transform surfaces.

In [ ]:
fig = visualize_decision_geometry(
    model=svm,
    X=wine.data,
    y=wine.target,
    projection_method="sensitivity",
    feature_names=list(wine.feature_names),
    class_names=dict(enumerate(wine.target_names)),
    show_confidence=True,
    show_centroids=True,
    show_ellipsoids=True,
    title="Test 2 — Wine SVM (Sensitivity: model-driven axes)",
)
fig.show()

---
## Test 3 — Breast Cancer GBM · Sensitivity (30-D features)
Checks: GBM + 30 named features + sensitivity projection.

In [ ]:
cancer = load_breast_cancer()
gbm = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=0)
gbm.fit(cancer.data, cancer.target)

fig = visualize_decision_geometry(
    model=gbm,
    X=cancer.data,
    y=cancer.target,
    projection_method="sensitivity",
    feature_names=list(cancer.feature_names),
    class_names={0: "Malignant", 1: "Benign"},
    show_confidence=True,
    show_centroids=True,
    show_ellipsoids=True,
    title="Test 3 — Breast Cancer GBM (Sensitivity, 30-D)",
)
fig.show()

---
## Test 4 — Latent Space · PCA / t-SNE / UMAP
Checks: `inspect_latent_space`, all three projection methods, ellipsoids, convex hulls.

In [ ]:
rng = np.random.default_rng(0)
D = 64
cluster_means = [
    np.zeros(D),
    np.array([8.0, 0.0, 0.0] + [0.0] * (D - 3)),
    np.array([0.0, 8.0, 0.0] + [0.0] * (D - 3)),
    np.array([4.0, 4.0, 6.0] + [0.0] * (D - 3)),
]
noise_std = np.full(D, 0.15)
noise_std[:3] = 0.7

embeddings = np.vstack([
    rng.normal(size=(150, D)) * noise_std + m
    for m in cluster_means
])
labels = np.repeat([0, 1, 2, 3], 150)
class_names = {0: "Science", 1: "Politics", 2: "Arts", 3: "Sport"}

cfg = DARK_SCIENTIFIC.copy()
cfg.projection.scale_input = False

for method in ("pca", "tsne", "umap"):
    print(f"  {method.upper()} ...")
    fig = inspect_latent_space(
        embeddings=embeddings,
        labels=labels,
        config=cfg.with_method(method),
        show_ellipsoids=True,
        show_convex_hulls=(method == "pca"),
        class_names=class_names,
        title=f"Test 4 — 64-D Embeddings ({method.upper()})",
    )
    fig.show()

print("All tests complete.")